# Early Experiments

See the repository README and data/README.md before running. Outputs were cleared for publication.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", str(ROOT / "data")))
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)


In [ ]:
import numpy as np 

In [ ]:
print( 'hello')

In [ ]:
import pandas as pd 


In [ ]:
costumer_features = pd.read_csv( str(DATA_DIR / "customer_features.csv"))
print( costumer_features.shape)

## DO the Complete PCA , Scaling and ploting the Graph

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt 

costumer_ids = costumer_features['client_id']

features = costumer_features.drop('client_id',axis=1  )
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print( features_scaled.shape)

In [ ]:
pca_temp = PCA()
pca_temp.fit(features_scaled)

# Explained variance plot
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(pca_temp.explained_variance_ratio_)+1),
         pca_temp.explained_variance_ratio_.cumsum(),
         marker='o', color='steelblue')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — How many components to keep?')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
cumsum = pca_temp.explained_variance_ratio_.cumsum()
for i, val in enumerate(cumsum):
    if val >= 0.90:
        print(f"90% variance at {i+1} components")
        break
for i, val in enumerate(cumsum):
    if val >= 0.97:
        print(f"97% variance at {i+1} components")
        break

In [ ]:
pca = PCA(n_components=min(13, *features_scaled.shape))

pca_results = pca.fit_transform( features_scaled)

print(f"Total variance explained: {pca.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
pca_df = pd.DataFrame(pca_results, columns=[f'PC{i+1}' for i in range(pca.n_components_)])
pca_df['client_id'] = costumer_ids.values

print("PCA DataFrame shape:", pca_df.shape)
pca_df.head(3)

In [ ]:
# Dependencies are installed using requirements.txt.

In [ ]:
import umap
reducer = umap.UMAP(n_components=4, random_state=42)
umap_result = reducer.fit_transform(pca_results)

print("UMAP output shape:", umap_result.shape)
plt.figure(figsize=(10, 7))
plt.scatter(umap_result[:, 1], umap_result[:, 2], 
            alpha=0.6, s=15, color='steelblue')
plt.title('UMAP — Customer Spending Behavior (2D)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()


In [ ]:
# Dependencies are installed using requirements.txt.

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=5)
cluster_labels = clusterer.fit_predict(umap_result)

print("Unique clusters:", len(set(cluster_labels[cluster_labels != -1])))
print("Noise points (-1):", (cluster_labels == -1).sum())
print("Cluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
from sklearn.metrics import silhouette_score

mask = cluster_labels != -1

score = silhouette_score(
    umap_result[mask],
    cluster_labels[mask]
)

print(score)

In [ ]:
umap_df = pd.DataFrame(umap_result, columns=['UMAP1', 'UMAP2' , 'UMAP3','UMAP4'])
umap_df['cluster'] = cluster_labels
umap_df['client_id'] = costumer_ids.values

# Colored UMAP plot
plt.figure(figsize=(10, 7))
colors = ['red' if c == -1 else plt.cm.tab10(c/7) for c in cluster_labels]
scatter = plt.scatter(umap_result[:, 3], umap_result[:, 0],
                      c=cluster_labels, cmap='tab10',
                      alpha=0.7, s=15)
plt.colorbar(scatter, label='Cluster')
plt.title('UMAP — Customer Clusters (HDBSCAN)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.show()

In [ ]:
costumer_features['cluster'] = cluster_labels

cluster_profile = costumer_features.groupby('cluster').agg({
    'avg_transaction_amount': 'mean',
    'total_spend'           : 'mean',
    'transaction_count'     : 'mean',
    'online_txn_ratio'      : 'mean',
    'swipe_txn_ratio'       : 'mean',
    'night_txn_ratio'       : 'mean',
    'weekend_txn_ratio'     : 'mean',
    'merchant_diversity'    : 'mean',
    'error_rate'            : 'mean',
    'spend_to_income_ratio' : 'mean',
}).round(3)

cluster_profile

In [ ]:
from sklearn.ensemble import IsolationForest


In [ ]:
fraud_per_customer = pd.read_csv( str(DATA_DIR / "fraud_check.csv"))
fraud_per_customer.head()

In [ ]:
validation = costumer_features[['client_id', 'anomaly']].merge(
    fraud_per_customer, on='client_id', how='left'
)
validation['has_fraud'] = validation['has_fraud'].fillna(0).astype(int)

In [ ]:
from sklearn.ensemble import IsolationForest

# Original scaled features pe run karo
iso_forest = IsolationForest(n_estimators=300, 
                              contamination=0.4, 
                              random_state=42)

anomaly_scores = iso_forest.fit_predict(features_scaled)
anomaly_score_values = iso_forest.decision_function(features_scaled)

# -1 = anomaly, 1 = normal
costumer_features['anomaly'] = anomaly_scores
costumer_features['anomaly_score'] = anomaly_score_values

print("Anomaly distribution:")
print(pd.Series(anomaly_scores).value_counts())
print(f"\nFlagged as fraud risk: {(anomaly_scores == -1).sum()} customers")



print("Validation Results:")
print(pd.crosstab(validation['anomaly'], validation['has_fraud'],
                  rownames=['Predicted'], colnames=['Actual Fraud']))

# Kitne actual fraud customers pakde
caught = validation[(validation['anomaly'] == -1) & (validation['has_fraud'] == 1)]
total_fraud_customers = validation['has_fraud'].sum()
print(f"\nTotal fraud customers    : {total_fraud_customers}")
print(f"Caught by Isolation Forest: {len(caught)}")
print(f"Detection rate           : {round(len(caught)/total_fraud_customers*100, 1)}%")
